In [1]:
!pip install pyLDAvis scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 7.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyLDAvis]1/2 [pyLDAvis]


In [2]:
import os
import sys
import re
# =========================================================
# 1. ÉP VERSION PYSPARK (PHẢI CHẠY ĐẦU TIÊN TRƯỚC KHI IMPORT)
# =========================================================
modules_to_remove = [mod for mod in sys.modules if mod.startswith('pyspark') or mod.startswith('py4j')]
for mod in modules_to_remove: 
    del sys.modules[mod]

sys.path = [p for p in sys.path if "/usr/local/spark" not in p]
if "PYTHONPATH" in os.environ: 
    del os.environ["PYTHONPATH"]
    
conda_site_packages = "/opt/conda/lib/python3.13/site-packages"
if conda_site_packages not in sys.path: 
    sys.path.insert(0, conda_site_packages)
    
os.environ["SPARK_HOME"] = os.path.join(conda_site_packages, "pyspark")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"


# =========================================================
# 2. BÂY GIỜ MỚI IMPORT PYSPARK VÀ KHỞI TẠO SESSION
# =========================================================
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, concat_ws
from pyspark.sql.types import IntegerType, StringType
from pyspark.ml.feature import StopWordsRemover, CountVectorizer
from pyspark.ml.clustering import LDA
spark = SparkSession.builder \
    .appName("pyLDAvis_Pipeline") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://raw-financial-data/iceberg_warehouse") \
    .getOrCreate()


# =========================================================
# 3. VÁ LỖI THỜI GIAN HADOOP
# =========================================================
hadoop_conf = spark._jsc.hadoopConfiguration()
iterator = hadoop_conf.iterator()
while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower()
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num, unit = int(match.group(1)), match.group(2)
        ms_val = num * 1000 if unit == 's' else num * 60000 if unit == 'm' else num * 3600000 if unit == 'h' else num * 86400000
        hadoop_conf.set(entry.getKey(), str(ms_val))

print("✅ Khởi tạo Spark và môi trường hoàn tất!")

✅ Khởi tạo Spark và môi trường hoàn tất!


In [7]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pyLDAvis
import pyLDAvis.lda_model
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
from pyspark.sql.functions import col, concat_ws

# =========================================================
# 1. KÉO DỮ LIỆU TỪ ICEBERG VỀ PYTHON (DRIVER)
# =========================================================
table_name = "my_catalog.processed_zone_finnhub.news_lda_summarized"
print("📥 1. Đang kéo 5000 bản tin mẫu từ Iceberg về RAM...")

# Lấy 5000 dòng từ cột 'title_tokens' để trình duyệt không bị quá tải khi vẽ hình
df_sample = spark.table(table_name).select("title_tokens").filter(col("title_tokens").isNotNull()).limit(10000)

# Cột title_tokens đang là mảng (Array), ta nối lại thành chuỗi (String) cách nhau bởi khoảng trắng
df_string = df_sample.withColumn("text_ready", concat_ws(" ", col("title_tokens")))

# Đưa dữ liệu thành danh sách (list) thuần túy của Python
docs = df_string.select("text_ready").rdd.flatMap(lambda x: x).collect()
print(f"✅ Đã kéo thành công {len(docs)} bản tin!")

📥 1. Đang kéo 5000 bản tin mẫu từ Iceberg về RAM...
✅ Đã kéo thành công 10000 bản tin!


In [8]:
# =========================================================
# 2. XÂY DỰNG MA TRẬN TỪ VỰNG VÀ HUẤN LUYỆN LDA
# =========================================================
print("🧮 2. Đang xây dựng Ma trận từ vựng (DTM)...")
# Giới hạn từ vựng: Bỏ qua các từ xuất hiện dưới 5 lần
tf_vectorizer = CountVectorizer(max_df=0.9, min_df=5)
dtm = tf_vectorizer.fit_transform(docs)

print("🧠 3. Đang mô phỏng thuật toán LDA (10 Chủ đề)...")
lda_model = LatentDirichletAllocation(
    n_components=10,        # Chia làm 10 chủ đề
    max_iter=10,            # Số vòng lặp tối đa
    learning_method='online', 
    random_state=42         # Giữ nguyên seed để biểu đồ không bị nhảy loạn xạ mỗi lần chạy lại
)
lda_model.fit(dtm)
print("✅ Huấn luyện hoàn tất!")

🧮 2. Đang xây dựng Ma trận từ vựng (DTM)...
🧠 3. Đang mô phỏng thuật toán LDA (10 Chủ đề)...
✅ Huấn luyện hoàn tất!


In [9]:
# =========================================================
# 3. KẾT XUẤT BIỂU ĐỒ PYLDAVIS
# =========================================================
print("🎨 4. Đang kết xuất biểu đồ tương tác pyLDAvis...")
pyLDAvis.enable_notebook()

# Dùng mds='tsne' để pyLDAvis phân bổ các vòng tròn ra xa nhau cho dễ nhìn
vis_data = pyLDAvis.lda_model.prepare(
    lda_model, 
    dtm, 
    tf_vectorizer, 
    mds='tsne'
)

# Hiển thị biểu đồ ngay trong Notebook
pyLDAvis.display(vis_data)

🎨 4. Đang kết xuất biểu đồ tương tác pyLDAvis...
